# Sensores IoT y detección de outliers
### Iván López

## Metodología
* Leer el archivo csv con PySpark.
* Agregar por sensor y hora
* Unir agregados con lecturas originales
* Marcar outliers
* Análisis de outliers

## Librerías requeridas

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Configuración de Spark

In [2]:
# Crear sesión Spark
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [4]:
# leer el CSV
df_sns = spark.read.csv('activity7_sensors_big.csv', header=True, inferSchema=True) 

In [5]:
df_sns = (df_sns.withColumn('reading_ts', F.to_timestamp('reading_ts')))

In [6]:
# verificar esquema
df_sns.printSchema()

root
 |-- sensor_id: integer (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- location: string (nullable = true)



In [7]:
#primeras filas
df_sns.show(10)

+---------+-------------------+-----------+--------+--------+
|sensor_id|         reading_ts|temperature|humidity|location|
+---------+-------------------+-----------+--------+--------+
|       42|2025-03-11 15:21:00|      20.08|    71.3|   Nave3|
|       51|2025-03-14 18:34:00|      22.36|   45.31|   Nave3|
|       50|2025-03-10 18:01:00|      24.65|   54.28|   Nave1|
|       96|2025-03-10 13:11:00|      25.08|   62.45|   Nave2|
|        8|2025-03-07 09:39:00|      24.31|   64.25|Exterior|
|       21|2025-03-07 01:00:00|       22.0|    52.7|Exterior|
|       51|2025-03-14 23:58:00|      21.11|   53.29|   Nave2|
|       17|2025-03-11 20:31:00|      25.68|   35.32|Exterior|
|       96|2025-03-14 09:17:00|      26.17|   58.96|   Nave2|
|        9|2025-03-07 14:10:00|      22.54|   39.39|   Nave1|
+---------+-------------------+-----------+--------+--------+
only showing top 10 rows


## Agregar por sensor y hora

In [8]:
#columna reading_hour
df_sns=(df_sns
        .withColumn('reading_hour', F.date_trunc('hour', 'reading_ts')))

df_sns.show(10)

+---------+-------------------+-----------+--------+--------+-------------------+
|sensor_id|         reading_ts|temperature|humidity|location|       reading_hour|
+---------+-------------------+-----------+--------+--------+-------------------+
|       42|2025-03-11 15:21:00|      20.08|    71.3|   Nave3|2025-03-11 15:00:00|
|       51|2025-03-14 18:34:00|      22.36|   45.31|   Nave3|2025-03-14 18:00:00|
|       50|2025-03-10 18:01:00|      24.65|   54.28|   Nave1|2025-03-10 18:00:00|
|       96|2025-03-10 13:11:00|      25.08|   62.45|   Nave2|2025-03-10 13:00:00|
|        8|2025-03-07 09:39:00|      24.31|   64.25|Exterior|2025-03-07 09:00:00|
|       21|2025-03-07 01:00:00|       22.0|    52.7|Exterior|2025-03-07 01:00:00|
|       51|2025-03-14 23:58:00|      21.11|   53.29|   Nave2|2025-03-14 23:00:00|
|       17|2025-03-11 20:31:00|      25.68|   35.32|Exterior|2025-03-11 20:00:00|
|       96|2025-03-14 09:17:00|      26.17|   58.96|   Nave2|2025-03-14 09:00:00|
|        9|2025-

In [9]:
df_agg=(df_sns.groupBy('sensor_id', 'reading_hour')
        .agg(F.round(F.avg('temperature'),2).alias('avg_temp'),
        F.round(F.stddev('temperature'),2).alias('std_temp'),
        F.count(F.lit(1)).alias('n_lectures'))
        .na.fill(0))

df_agg.show(10)

+---------+-------------------+--------+--------+----------+
|sensor_id|       reading_hour|avg_temp|std_temp|n_lectures|
+---------+-------------------+--------+--------+----------+
|       21|2025-03-13 16:00:00|   20.33|     0.0|         1|
|       80|2025-03-13 14:00:00|   23.16|     0.0|         1|
|       93|2025-03-06 02:00:00|   27.35|    2.96|         3|
|       13|2025-03-01 07:00:00|   23.67|     0.0|         1|
|       65|2025-03-10 08:00:00|   27.09|    1.39|         2|
|        5|2025-03-07 05:00:00|   25.87|    2.46|         4|
|       81|2025-03-12 23:00:00|   25.22|    1.04|         3|
|       36|2025-03-02 06:00:00|   22.71|    5.28|         2|
|       10|2025-03-04 02:00:00|   24.46|    0.87|         4|
|       32|2025-03-11 21:00:00|   25.07|     0.6|         3|
+---------+-------------------+--------+--------+----------+
only showing top 10 rows


## Unir agregados con lecturas originales

In [10]:
df_join = df_sns.join(
        df_agg,
        on=['sensor_id', 'reading_hour'], 
        how='left'                         
        )

df_join.show(10)

+---------+-------------------+-------------------+-----------+--------+--------+--------+--------+----------+
|sensor_id|       reading_hour|         reading_ts|temperature|humidity|location|avg_temp|std_temp|n_lectures|
+---------+-------------------+-------------------+-----------+--------+--------+--------+--------+----------+
|       42|2025-03-11 15:00:00|2025-03-11 15:21:00|      20.08|    71.3|   Nave3|   20.08|     0.0|         1|
|       51|2025-03-14 18:00:00|2025-03-14 18:34:00|      22.36|   45.31|   Nave3|   23.94|    2.53|         3|
|       50|2025-03-10 18:00:00|2025-03-10 18:01:00|      24.65|   54.28|   Nave1|   24.37|    2.67|         4|
|       96|2025-03-10 13:00:00|2025-03-10 13:11:00|      25.08|   62.45|   Nave2|   25.08|     0.0|         1|
|        8|2025-03-07 09:00:00|2025-03-07 09:39:00|      24.31|   64.25|Exterior|   24.31|     0.0|         1|
|       21|2025-03-07 01:00:00|2025-03-07 01:00:00|       22.0|    52.7|Exterior|   25.27|    2.59|         4|
|

## Marcar outliers

In [11]:
df_join = (df_join
            .withColumn('temperature', F.round(F.col('temperature'), 2))
            .withColumn('is_outlier',
            (F.col('temperature') > (F.col('avg_temp') + 3 * F.col('std_temp'))) | 
            (F.col('temperature') < (F.col('avg_temp') - 3 * F.col('std_temp')))
            ))

df_join.show(10)

+---------+-------------------+-------------------+-----------+--------+--------+--------+--------+----------+----------+
|sensor_id|       reading_hour|         reading_ts|temperature|humidity|location|avg_temp|std_temp|n_lectures|is_outlier|
+---------+-------------------+-------------------+-----------+--------+--------+--------+--------+----------+----------+
|       42|2025-03-11 15:00:00|2025-03-11 15:21:00|      20.08|    71.3|   Nave3|   20.08|     0.0|         1|     false|
|       51|2025-03-14 18:00:00|2025-03-14 18:34:00|      22.36|   45.31|   Nave3|   23.94|    2.53|         3|     false|
|       50|2025-03-10 18:00:00|2025-03-10 18:01:00|      24.65|   54.28|   Nave1|   24.37|    2.67|         4|     false|
|       96|2025-03-10 13:00:00|2025-03-10 13:11:00|      25.08|   62.45|   Nave2|   25.08|     0.0|         1|     false|
|        8|2025-03-07 09:00:00|2025-03-07 09:39:00|      24.31|   64.25|Exterior|   24.31|     0.0|         1|     false|
|       21|2025-03-07 01

In [12]:
df_outliers = df_join.filter(F.col('is_outlier') == True)

df_outliers.show(10)

+---------+------------+----------+-----------+--------+--------+--------+--------+----------+----------+
|sensor_id|reading_hour|reading_ts|temperature|humidity|location|avg_temp|std_temp|n_lectures|is_outlier|
+---------+------------+----------+-----------+--------+--------+--------+--------+----------+----------+
+---------+------------+----------+-----------+--------+--------+--------+--------+----------+----------+



# Resumen de outliers

### Por sensor

In [17]:
(df_outliers.groupBy('sensor_id')
            .agg(F.count(F.lit(1)).alias('outliers_count'))
            .orderBy(F.desc('outliers_count'))
            .show())

+---------+--------------+
|sensor_id|outliers_count|
+---------+--------------+
+---------+--------------+



### Por ubicación

In [19]:
(df_outliers.groupBy('location')
            .agg(F.count(F.lit(1)).alias('outliers_count'))
            .orderBy(F.desc('outliers_count'))
            .show())

+--------+--------------+
|location|outliers_count|
+--------+--------------+
+--------+--------------+



## Análisis
No hay outliers de mediciones de temperatura por las lecturas de los sensores dentro de una misma hora en todo el dataset, por lo que tampoco los hay al hacer distinción por sensor o ubicación. 

Sin embargo, realizar el análisis en este contexto puede ser de utilidad para que cuando sí los haya, se puedan identificar fallas técnicas momentáneas como ruido eléctrico o batería baja, descalibración o desgaste del sensor, así como eventos críticos, por ejemplo incendios, puertas abiertas o fallas en motores de refrigeración para la industria. 

Para hacer esta distinción, se suele observar que si los outliers son aislados en un solo sensor esto sugiere error del dispositivo, pero si múltiples sensores en la misma zona detectan outliers simultáneamente, es muy probable que se trate de una incidencia que requiere atención inmediata.